<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup → Data → Excel Grid Logic → V0 Grid → 1m Backtest → Cash Movement & Excel Grid Cashflow → Audit → Results

## 1. Setup & Data

In [ ]:
import os, io, glob, datetime, heapq, bisect
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR='/content/drive/MyDrive/03.Trading/00.Live Trading'
SYMBOL='BTCUSDT'; START_DATE='2024-01-01'; END_DATE='2026-01-01'
BUY_FEE=0.001; SELL_FEE=0.001
GRID_CAPITAL=3000.0; GRID_CEILING=8987.0; GRID_FLOOR=1987.0; GRID_GAP=70.0

def load_market_data(symbol,timeframe,data_dir):
    p=os.path.join(data_dir,f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(p): raise FileNotFoundError(p)
    df=pd.read_csv(p); df['open_time']=pd.to_datetime(df['open_time'],utc=True)
    df[['open','high','low','close','volume']]=df[['open','high','low','close','volume']].astype(float)
    return df.drop_duplicates('open_time').sort_values('open_time').reset_index(drop=True)

df_1m=load_market_data(SYMBOL,'1m',DATA_DIR)
start_ts=pd.Timestamp(START_DATE,tz='UTC'); end_ts=pd.Timestamp(END_DATE,tz='UTC')
df_1m=df_1m.loc[(df_1m.open_time>=start_ts)&(df_1m.open_time<end_ts)].reset_index(drop=True)
print(f'Rows: {len(df_1m):,} | {df_1m.open_time.min()} -> {df_1m.open_time.max()}')

## 2. Excel Grid Logic

The KZM Excel template is the source of truth for fixed-grid formulas.

In [ ]:
def build_excel_grid_table(capital,ceiling,floor,gap,buy_fee=0.001,sell_fee=0.001):
    if capital<=0 or ceiling<=floor or gap<=0: raise ValueError('Invalid grid inputs.')
    raw=(ceiling-floor)/gap
    if not np.isclose(raw,round(raw)): raise ValueError('(ceiling-floor) must be divisible by gap.')
    n=int(round(raw)); cost=capital/n
    buy=ceiling-gap*np.arange(1,n+1); sell=buy+gap
    gross_base=cost/buy; buy_fee_base=gross_base*buy_fee; base=gross_base-buy_fee_base
    gross_sell=base*sell; sell_fee_quote=gross_sell*sell_fee; net_sell=gross_sell-sell_fee_quote
    return pd.DataFrame({'level':np.arange(1,n+1),'buy_price':buy,'sell_price':sell,'capital_per_level':cost,
        'gross_base_amount':gross_base,'buy_fee_base':buy_fee_base,'base_amount':base,'gross_sell':gross_sell,
        'sell_fee_quote':sell_fee_quote,'net_sell':net_sell,'profit':net_sell-cost})

df_grid_excel=build_excel_grid_table(GRID_CAPITAL,GRID_CEILING,GRID_FLOOR,GRID_GAP,BUY_FEE,SELL_FEE)
x=df_grid_excel.iloc[0]
assert np.isclose(x.profit,0.1750644398340242,atol=1e-12)
print(f'Excel checkpoint PASSED: {x.buy_price:.0f} -> {x.sell_price:.0f}, profit={x.profit:.6f}')

## 3. Historical Grid Configuration — V0

Capital and Gap are strategy inputs like Excel. Full-period Low/High are used only to select V0 boundaries, so this version has look-ahead bias and is for execution validation only.

In [ ]:
BACKTEST_CAPITAL=3000.0; BACKTEST_GAP=1000.0; PRICE_ROUNDING=1000.0
historical_low=df_1m.low.min(); historical_high=df_1m.high.max()
BACKTEST_FLOOR=np.floor(historical_low/PRICE_ROUNDING)*PRICE_ROUNDING
BACKTEST_CEILING=np.ceil(historical_high/PRICE_ROUNDING)*PRICE_ROUNDING
NUMBER_OF_GRIDS=int(round((BACKTEST_CEILING-BACKTEST_FLOOR)/BACKTEST_GAP))
CAPITAL_PER_LEVEL=BACKTEST_CAPITAL/NUMBER_OF_GRIDS
df_grid_backtest=build_excel_grid_table(BACKTEST_CAPITAL,BACKTEST_CEILING,BACKTEST_FLOOR,BACKTEST_GAP,BUY_FEE,SELL_FEE)
print('===== V0 Backtest Grid Configuration =====')
print(f'Historical Low     : {historical_low:,.2f} USDT')
print(f'Historical High    : {historical_high:,.2f} USDT')
print(f'Grid Floor         : {BACKTEST_FLOOR:,.2f} USDT')
print(f'Grid Ceiling       : {BACKTEST_CEILING:,.2f} USDT')
print(f'Grid Gap           : {BACKTEST_GAP:,.2f} USDT')
print(f'Number of Grids    : {NUMBER_OF_GRIDS}')
print(f'Capital / Grid     : {CAPITAL_PER_LEVEL:,.2f} USDT')

## 4. Backtest Engine — 1 Minute

BUY occurs only on a downward crossing. Existing SELL targets may fill on candle High. A new BUY cannot SELL in the same candle, a sold grid cannot rebuy in the same candle, and same-candle SELL proceeds are not reused for BUYs.

In [ ]:
def run_grid_backtest(df_price,grid_table,initial_capital):
    req={'open_time','open','high','low','close'}; missing=req-set(df_price.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    if len(df_price)==0 or initial_capital<=0: raise ValueError('Invalid backtest input.')
    data=df_price.sort_values('open_time').reset_index(drop=True)
    g=grid_table.sort_values('buy_price').reset_index(drop=True).copy()
    buy=g.buy_price.to_numpy(float); sell=g.sell_price.to_numpy(float); cost=g.capital_per_level.to_numpy(float)
    base=g.base_amount.to_numpy(float); bf=g.buy_fee_base.to_numpy(float); sf=g.sell_fee_quote.to_numpy(float)
    net=g.net_sell.to_numpy(float); profit=g.profit.to_numpy(float); lvl=g.level.to_numpy(int)
    if len(g)>1 and not np.allclose(np.diff(buy),np.diff(buy)[0]): raise ValueError('Arithmetic grid required.')
    holding=np.zeros(len(g),bool); buy_time=[None]*len(g); heap=[]
    cash=float(initial_capital); btc=0.0; realized=0.0; buy_fee_btc=buy_fee_eq=sell_fee=0.0; cycles=0
    events=[]; completed=[]; n=len(data); eq=np.empty(n); cash_v=np.empty(n); btc_v=np.empty(n)
    buy_list=buy.tolist(); prev_close=None; event_id=0
    for i,row in enumerate(data.itertuples(index=False)):
        t=row.open_time; op=float(row.open); hi=float(row.high); lo=float(row.low); cl=float(row.close)
        cash_start=cash; sold=set()
        while heap and heap[0][0]<=hi:
            _,k=heapq.heappop(heap)
            if not holding[k]: continue
            cb=cash; bb=btc; holding[k]=False; cash+=net[k]; btc-=base[k]
            if abs(btc)<1e-12: btc=0.0
            realized+=profit[k]; sell_fee+=sf[k]; cycles+=1; sold.add(k); event_id+=1
            completed.append({'grid_level':int(lvl[k]),'buy_time':buy_time[k],'sell_time':t,'buy_price':buy[k],
                'sell_price':sell[k],'cost':cost[k],'base_amount':base[k],'actual_earn':net[k],
                'grid_cashflow':profit[k],'profit':profit[k]})
            events.append({'event_id':event_id,'time':t,'side':'SELL','grid_level':int(lvl[k]),'price':sell[k],
                'base_amount':base[k],'quote_amount':net[k],'fee_base':0.0,'fee_quote':sf[k],
                'realized_profit':profit[k],'cash_movement':net[k],'grid_cashflow':profit[k],
                'cash_before':cb,'cash_after':cash,'btc_before':bb,'btc_after':btc})
            buy_time[k]=None
        budget=cash_start; down_start=op if prev_close is None else max(prev_close,op)
        if lo<down_start:
            a=bisect.bisect_left(buy_list,lo); b=bisect.bisect_left(buy_list,down_start)
            for k in range(b-1,a-1,-1):
                if holding[k] or k in sold: continue
                if budget+1e-12<cost[k]: break
                cb=cash; bb=btc; holding[k]=True; buy_time[k]=t; budget-=cost[k]; cash-=cost[k]; btc+=base[k]
                buy_fee_btc+=bf[k]; buy_fee_eq+=bf[k]*buy[k]; heapq.heappush(heap,(sell[k],k)); event_id+=1
                events.append({'event_id':event_id,'time':t,'side':'BUY','grid_level':int(lvl[k]),'price':buy[k],
                    'base_amount':base[k],'quote_amount':cost[k],'fee_base':bf[k],'fee_quote':0.0,
                    'realized_profit':0.0,'cash_movement':-cost[k],'grid_cashflow':0.0,
                    'cash_before':cb,'cash_after':cash,'btc_before':bb,'btc_after':btc})
        eq[i]=cash+btc*cl; cash_v[i]=cash; btc_v[i]=btc; prev_close=cl
    curve=pd.DataFrame({'open_time':data.open_time.to_numpy(),'close':data.close.to_numpy(float),'cash':cash_v,'btc':btc_v,'equity':eq})
    peak=np.maximum.accumulate(eq); dd=eq/peak-1; curve['drawdown']=dd
    final=float(eq[-1]); maxdd=float(dd.min()); netret=final/initial_capital-1
    days=(data.open_time.iloc[-1]-data.open_time.iloc[0]).total_seconds()/86400
    ann=np.nan
    if days>0 and final>0:
        growth=np.log(final/initial_capital)*(365.25/days)
        if growth<700: ann=float(np.expm1(growth))
    calmar=float(ann/abs(maxdd)) if maxdd<0 and np.isfinite(ann) else np.nan
    log=pd.DataFrame(events)
    if not log.empty:
        log['cumulative_cash_movement']=log.cash_movement.cumsum(); log['cumulative_grid_cashflow']=log.grid_cashflow.cumsum()
    summary={'initial_capital':float(initial_capital),'final_equity':final,'net_return':float(netret),'annualized_return':ann,
        'max_drawdown':maxdd,'calmar_ratio':calmar,'completed_cycles':int(cycles),'open_positions':int(holding.sum()),
        'final_cash':float(cash),'final_btc':float(btc),'realized_profit':float(realized),
        'unrealized_pnl':float(final-initial_capital-realized),'buy_fee_btc':float(buy_fee_btc),
        'buy_fee_usdt_equiv':float(buy_fee_eq),'sell_fee_usdt':float(sell_fee),'total_fee_usdt_equiv':float(buy_fee_eq+sell_fee)}
    return {'summary':summary,'trade_log':log,'completed_trades':pd.DataFrame(completed),'equity_curve':curve,
        'grid_state':g.assign(holding=holding,buy_time=buy_time)}

In [ ]:
backtest_result=run_grid_backtest(df_1m,df_grid_backtest,BACKTEST_CAPITAL)
backtest_summary=backtest_result['summary']; df_trade_log=backtest_result['trade_log']
df_completed_trades=backtest_result['completed_trades']; df_equity_curve=backtest_result['equity_curve']; df_grid_state=backtest_result['grid_state']
print('Backtest completed.')

## 5. Cash Movement & Excel Grid Cashflow

**Cash Movement** is actual USDT entering/leaving the cash account, so BUY is negative and SELL is positive. **Grid Cashflow (Excel)** is `Actual Earn - Cost` for a completed grid cycle; BUY is 0 and the value is recognized at SELL. This keeps the Excel meaning separate from the cash-account ledger.

In [ ]:
df_cash_ledger=df_trade_log[['event_id','time','side','grid_level','price','cash_movement','cumulative_cash_movement',
    'cash_before','cash_after','base_amount','btc_before','btc_after','fee_base','fee_quote']].copy()
df_grid_cashflow=df_completed_trades[['grid_level','buy_time','sell_time','buy_price','sell_price','cost','actual_earn','grid_cashflow']].copy()
df_grid_cashflow['cumulative_grid_cashflow']=df_grid_cashflow.grid_cashflow.cumsum()
print('===== Cash Movement Reconciliation =====')
cm=df_cash_ledger.cash_movement.sum()
print(f'Initial Capital       : {BACKTEST_CAPITAL:,.2f} USDT')
print(f'Net Cash Movement     : {cm:,.2f} USDT')
print(f'Expected Final Cash   : {BACKTEST_CAPITAL+cm:,.2f} USDT')
print(f'Backtest Final Cash   : {backtest_summary["final_cash"]:,.2f} USDT')
print('\n===== Excel Grid Cashflow Reconciliation =====')
print(f'Completed Cycles      : {len(df_grid_cashflow):,}')
print(f'Total Grid Cashflow   : {df_grid_cashflow.grid_cashflow.sum():,.2f} USDT')
print(f'Realized Profit       : {backtest_summary["realized_profit"]:,.2f} USDT')
display(df_grid_cashflow.head(20)); display(df_cash_ledger.head(20))

## 6. System Audit

Full-run consistency checks are separate from the synthetic **Unit Tests** in `tests/test_grid_trading.py`.

In [ ]:
def build_system_test_log(summary,trade_log,completed_trades,equity_curve,grid_state,grid_table,initial_capital,gap):
    tests=[]
    def add(name,ok,expected,actual): tests.append({'test':name,'status':'PASS' if ok else 'FAIL','expected':expected,'actual':actual})
    add('Grid spacing = configured Gap',bool(np.allclose(grid_table.sell_price-grid_table.buy_price,gap)),gap,'all levels')
    e=(trade_log.cash_before+trade_log.cash_movement-trade_log.cash_after).abs().max() if len(trade_log) else 0
    add('Cash movement identity per event',e<=1e-9,'error <= 1e-9',e)
    net=trade_log.cash_movement.sum() if len(trade_log) else 0; expected_cash=initial_capital+net
    add('Initial cash + net movement = final cash',abs(expected_cash-summary['final_cash'])<=1e-8,expected_cash,summary['final_cash'])
    gc=trade_log.grid_cashflow.sum() if len(trade_log) else 0
    add('Excel Grid Cashflow = realized profit',abs(gc-summary['realized_profit'])<=1e-8,summary['realized_profit'],gc)
    buys=int(trade_log.side.eq('BUY').sum()); sells=int(trade_log.side.eq('SELL').sum())
    add('BUY - SELL = open positions',buys-sells==summary['open_positions'],buys-sells,summary['open_positions'])
    add('SELL count = completed cycles',sells==summary['completed_cycles'],sells,summary['completed_cycles'])
    same=int((pd.to_datetime(completed_trades.sell_time,utc=True)<=pd.to_datetime(completed_trades.buy_time,utc=True)).sum()) if len(completed_trades) else 0
    add('SELL occurs after BUY minute',same==0,0,same)
    eqerr=(equity_curve.cash+equity_curve.btc*equity_curve.close-equity_curve.equity).abs().max()
    add('Cash + BTC x Close = Equity',eqerr<=1e-8,'error <= 1e-8',eqerr)
    add('Cash never negative',equity_curve.cash.min()>=-1e-9,'>= 0',equity_curve.cash.min())
    held=int(grid_state.holding.sum()); add('Held grid count = open positions',held==summary['open_positions'],held,summary['open_positions'])
    held_btc=float(grid_state.loc[grid_state.holding,'base_amount'].sum())
    add('Held BTC = final BTC',abs(held_btc-summary['final_btc'])<=1e-10,held_btc,summary['final_btc'])
    pnl=summary['realized_profit']+summary['unrealized_pnl']; gain=summary['final_equity']-summary['initial_capital']
    add('Realized + unrealized = equity gain',abs(pnl-gain)<=1e-8,gain,pnl)
    return pd.DataFrame(tests)

df_system_test_log=build_system_test_log(backtest_summary,df_trade_log,df_completed_trades,df_equity_curve,df_grid_state,
    df_grid_backtest,BACKTEST_CAPITAL,BACKTEST_GAP)
display(df_system_test_log)
failed=df_system_test_log[df_system_test_log.status.eq('FAIL')]
assert failed.empty,'SYSTEM AUDIT FAILED:\n'+failed.to_string(index=False)
print('SYSTEM AUDIT: ALL TESTS PASSED')

## 7. Results

In [ ]:
s=backtest_summary
print('===== V0 Backtest Summary =====')
print(f'Initial Capital     : {s["initial_capital"]:,.2f} USDT')
print(f'Final Equity        : {s["final_equity"]:,.2f} USDT')
print(f'Net Return          : {s["net_return"]:.2%}')
print(f'Annualized Return   : {s["annualized_return"]:.2%}')
print(f'Max Drawdown        : {s["max_drawdown"]:.2%}')
print(f'Calmar Ratio        : {s["calmar_ratio"]:.3f}')
print(f'Completed Cycles    : {s["completed_cycles"]:,}')
print(f'Open Positions      : {s["open_positions"]:,}')
print(f'Final Cash          : {s["final_cash"]:,.2f} USDT')
print(f'Final BTC           : {s["final_btc"]:.8f} BTC')
print(f'Realized Profit     : {s["realized_profit"]:,.2f} USDT')
print(f'Unrealized P&L      : {s["unrealized_pnl"]:,.2f} USDT')
print(f'Total Fee (USDT eq.): {s["total_fee_usdt_equiv"]:,.2f} USDT')

daily=df_equity_curve.set_index('open_time').equity.resample('1D').last().dropna()
plt.figure(figsize=(14,5)); plt.plot(daily.index,daily.values); plt.title('BTC Spot Fixed Grid — Daily Portfolio Equity')
plt.xlabel('Date'); plt.ylabel('Equity (USDT)'); plt.grid(True,alpha=.3); plt.show()